<a href="https://colab.research.google.com/github/VaggelisApostolou/Challenges-in-Detecting-Toxic-Language-in-Greek-Sports-Social-Media/blob/main/ModelsByLanguage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports**

In [ ]:
!pip install --upgrade transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import os, random, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import math
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    XLMRobertaTokenizerFast,
    XLMRobertaConfig,
    XLMRobertaForSequenceClassification,
    XLMRobertaForMaskedLM,
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, classification_report, precision_score, recall_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
XLM_CHECKPOINT = "xlm-roberta-base"
GREEKB_CHECKPOINT = "nlpaueb/bert-base-greek-uncased-v1"
MMB_CHECKPOINT = "jhu-clsp/mmBERT-base"
MASKED_XLM = "/content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model"
MASKED_GREEKB = "/content/drive/MyDrive/Toxic language in football Research/models/Greek-BERT"
MASKED_MMB = "/content/drive/MyDrive/Toxic language in football Research/models/mmBERT"

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Toxic language in football Research/final_labeled_batches/final_labeled_language.csv"

# **Dataset Analysis**

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
df_en = df[df['language'] == 'en']
df_el = df[df['language'] == 'el']

# **Models in English Dataset**

## *Configuration*

In [ ]:
SEED      = 42
EPOCHS    = 8
BATCH_SZ  = 32
LR        = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_LEN   = 256
PATIENCE  = 3
GRAD_CLIP = 1.0

LOSS_TYPE = "focal"   # "ce" or "focal"
FOCAL_GAMMA = 2.0

# Toxic class emphasis
TOXIC_WEIGHT_MULT = 1.5
USE_SAMPLER = True

# Threshold objective
FBETA = 2.0              # tune threshold on F_beta (beta=2 -> recall emphasis)

set_seed(SEED)
print("Device:", device)
print("Loss:", LOSS_TYPE, "| gamma:", FOCAL_GAMMA, "| toxic_w_mult:", TOXIC_WEIGHT_MULT, "| Fbeta:", FBETA)


Device: cuda
Loss: focal | gamma: 2.0 | toxic_w_mult: 1.5 | Fbeta: 2.0


## *Load Dataset*

In [ ]:
def load_dataset(path: str):
    df = df_en
    if not {"text", "label"}.issubset(df.columns):
        if df.shape[1] == 2:
            df.columns = ["text", "label"]
        else:
            raise ValueError(f"Το CSV πρέπει να έχει στήλες 'text' και 'label'. Βρέθηκαν: {df.columns.tolist()}")

    df = df.dropna(subset=["label", "text"]).copy()

    label2id = {"Non-toxic": 0, "Toxic": 1}

    if not set(df["label"].unique()).issubset(set(label2id.keys())):
        raise ValueError(f"Τα labels πρέπει να είναι {list(label2id.keys())}, βρέθηκαν: {df['label'].unique()}")

    df["label_id"] = df["label"].map(label2id)

    return df, label2id

In [ ]:
df, label2id = load_dataset(DATA_PATH)
print("Total shape:", df.shape)
print("Total label distribution:\n", df["label"].value_counts(normalize=True).round(3))
df.head()

Total shape: (4999, 5)
Total label distribution:
 label
Non-toxic    0.944
Toxic        0.056
Name: proportion, dtype: float64


,id,text,label,language,label_id
0,3438,@user @user I miss you Valbuş💛💙 Come to İstanb...,Non-toxic,en,0
1,3439,@user We're extremely glad to see y'all landed...,Non-toxic,en,0
2,3440,@user Lucky ass goal,Non-toxic,en,0
3,3441,@user Your deeds for the motherland won't go u...,Non-toxic,en,0
5,3443,@user not nearly good enough for the jersey so...,Toxic,en,1


## *Train-Test Split*

In [ ]:
# First: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
# Then: temp -> val (15%) and test (15%) i.e., split temp 50/50 stratified
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

def pct(x): return (x.value_counts(normalize=True).rename({0:"non-toxic",1:"toxic"})*100).round(1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist (%):\n", pct(train_df["label_id"]))
print("Val   dist (%):\n", pct(val_df["label_id"]))
print("Test  dist (%):\n", pct(test_df["label_id"]))

Split sizes: 3499 750 750
Train dist (%):
 label_id
non-toxic    94.4
toxic         5.6
Name: proportion, dtype: float64
Val   dist (%):
 label_id
non-toxic    94.4
toxic         5.6
Name: proportion, dtype: float64
Test  dist (%):
 label_id
non-toxic    94.5
toxic         5.5
Name: proportion, dtype: float64


## *Class Weights*

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=train_df["label_id"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights[1] = class_weights[1] * TOXIC_WEIGHT_MULT  # boost toxic
class_weights = class_weights.to(device)
print("Class weights (train) with boost:", class_weights.tolist())

Class weights (train) with boost: [0.5295096635818481, 13.457693099975586]


## *Focal Loss*

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum": return loss.sum()
        return loss

if LOSS_TYPE.lower() == "focal":
    loss_fn = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
else:
    loss_fn = lambda logits, targets: F.cross_entropy(logits, targets, weight=class_weights)
loss_fn

FocalLoss()

# **Model 1 (XLM-RoBERTa)**

## **Classic**

In [ ]:
MODEL_NAME = XLM_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 2.2650 | ValAcc 0.0560 | ValF1_macro 0.0530 | ValF1_toxic 0.1061 | ValPrec_toxic 0.0560 | ValRec_toxic 1.0000
Val AUROC: 0.9072
Epoch  2 | TrainLoss 0.2568 | ValAcc 0.8493 | ValF1_macro 0.6548 | ValF1_toxic 0.3957 | ValPrec_toxic 0.2552 | ValRec_toxic 0.8810
Val AUROC: 0.9250
Epoch  3 | TrainLoss 0.1856 | ValAcc 0.9080 | ValF1_macro 0.7154 | ValF1_toxic 0.4812 | ValPrec_toxic 0.3516 | ValRec_toxic 0.7619
Val AUROC: 0.8351
Epoch  4 | TrainLoss 0.1148 | ValAcc 0.9467 | ValF1_macro 0.7586 | ValF1_toxic 0.5455 | ValPrec_toxic 0.5217 | ValRec_toxic 0.5714
Val AUROC: 0.9098
Epoch  5 | TrainLoss 0.1358 | ValAcc 0.9467 | ValF1_macro 0.7684 | ValF1_toxic 0.5652 | ValPrec_toxic 0.5200 | ValRec_toxic 0.6190
Val AUROC: 0.9459
Epoch  6 | TrainLoss 0.0642 | ValAcc 0.9547 | ValF1_macro 0.7903 | ValF1_toxic 0.6047 | ValPrec_toxic 0.5909 | ValRec_toxic 0.6190
Val AUROC: 0.9377
Epoch  7 | TrainLoss 0.1033 | ValAcc 0.9573 | ValF1_macro 0.7936 | ValF1_toxic 0.6098 | ValPrec_toxic 0.62

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.95      0.97       708
           1       0.49      0.74      0.59        42

    accuracy                           0.94       750
   macro avg       0.74      0.85      0.78       750
weighted avg       0.96      0.94      0.95       750


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9573 | Val Macro-F1: 0.7936 | Val F1-toxic: 0.6098
Best thr=0.100 (beta=2.0) -> Val Acc: 0.9427 | Val Macro-F1: 0.7798 | Val F1-toxic: 0.5905
Val AUROC: 0.9419
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       709
           1       0.29      0.22      0.25        41

    accuracy                           0.93       750
   macro avg       0.62      0.59      0.61       750
weighted avg       0.92      0.93      0.92       750

              precision    recall  f1-score   support

           0       0.96      0.93      0.95       709
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_XLM

### *Model Setup*

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)
config = XLMRobertaConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/my_domain_adapted_model
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 2.1008 | ValAcc 0.0560 | ValF1_macro 0.0530 | ValF1_toxic 0.1061 | ValPrec_toxic 0.0560 | ValRec_toxic 1.0000
Val AUROC: 0.8748
Epoch  2 | TrainLoss 0.2898 | ValAcc 0.8867 | ValF1_macro 0.6942 | ValF1_toxic 0.4516 | ValPrec_toxic 0.3097 | ValRec_toxic 0.8333
Val AUROC: 0.9100
Epoch  3 | TrainLoss 0.1637 | ValAcc 0.9360 | ValF1_macro 0.7564 | ValF1_toxic 0.5472 | ValPrec_toxic 0.4531 | ValRec_toxic 0.6905
Val AUROC: 0.9196
Epoch  4 | TrainLoss 0.1391 | ValAcc 0.9493 | ValF1_macro 0.7886 | ValF1_toxic 0.6042 | ValPrec_toxic 0.5370 | ValRec_toxic 0.6905
Val AUROC: 0.9036
Epoch  5 | TrainLoss 0.1018 | ValAcc 0.9373 | ValF1_macro 0.7593 | ValF1_toxic 0.5524 | ValPrec_toxic 0.4603 | ValRec_toxic 0.6905
Val AUROC: 0.9288
Epoch  6 | TrainLoss 0.0678 | ValAcc 0.9440 | ValF1_macro 0.7663 | ValF1_toxic 0.5625 | ValPrec_toxic 0.5000 | ValRec_toxic 0.6429
Val AUROC: 0.9429
Epoch  7 | TrainLoss 0.0338 | ValAcc 0.9480 | ValF1_macro 0.7809 | ValF1_toxic 0.5895 | ValPrec_toxic 0.52

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.94      0.96       708
           1       0.42      0.79      0.55        42

    accuracy                           0.93       750
   macro avg       0.70      0.86      0.76       750
weighted avg       0.96      0.93      0.94       750


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9493 | Val Macro-F1: 0.7886 | Val F1-toxic: 0.6042
Best thr=0.103 (beta=2.0) -> Val Acc: 0.9280 | Val Macro-F1: 0.7554 | Val F1-toxic: 0.5500
Val AUROC: 0.9036
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       709
           1       0.41      0.51      0.46        41

    accuracy                           0.93       750
   macro avg       0.69      0.73      0.71       750
weighted avg       0.94      0.93      0.94       750

              precision    recall  f1-score   support

           0       0.97      0.92      0.95       709
     

# **Model 2 (mmBERT)**

## **Classic**

In [ ]:
MODEL_NAME = MMB_CHECKPOINT

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Epoch  1 | TrainLoss 0.9386 | ValAcc 0.6587 | ValF1_macro 0.5111 | ValF1_toxic 0.2426 | ValPrec_toxic 0.1385 | ValRec_toxic 0.9762
Val AUROC: 0.9503
Epoch  2 | TrainLoss 0.0924 | ValAcc 0.9453 | ValF1_macro 0.7937 | ValF1_toxic 0.6168 | ValPrec_toxic 0.5077 | ValRec_toxic 0.7857
Val AUROC: 0.9634
Epoch  3 | TrainLoss 0.0343 | ValAcc 0.9587 | ValF1_macro 0.7708 | ValF1_toxic 0.5634 | ValPrec_toxic 0.6897 | ValRec_toxic 0.4762
Val AUROC: 0.9657
Epoch  4 | TrainLoss 0.0127 | ValAcc 0.9587 | ValF1_macro 0.7825 | ValF1_toxic 0.5867 | ValPrec_toxic 0.6667 | ValRec_toxic 0.5238
Val AUROC: 0.9520
Epoch  5 | TrainLoss 0.0064 | ValAcc 0.9627 | ValF1_macro 0.7843 | ValF1_toxic 0.5882 | ValPrec_toxic 0.7692 | ValRec_toxic 0.4762
Val AUROC: 0.9502
Epoch  6 | TrainLoss 0.0001 | ValAcc 0.9613 | ValF1_macro 0.7856 | ValF1_toxic 0.5915 | ValPrec_toxic 0.7241 | ValRec_toxic 0.5000
Val AUROC: 0.9476
Early stopping at epoch 6. Best Val macro-F1: 0.7937
Loaded best checkpoint (Val macro-F1=0.7937)


### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       708
           1       0.59      0.76      0.67        42

    accuracy                           0.96       750
   macro avg       0.79      0.87      0.82       750
weighted avg       0.96      0.96      0.96       750


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9453 | Val Macro-F1: 0.7937 | Val F1-toxic: 0.6168
Best thr=0.698 (beta=2.0) -> Val Acc: 0.9573 | Val Macro-F1: 0.8219 | Val F1-toxic: 0.6667
Val AUROC: 0.9634
              precision    recall  f1-score   support

           0       0.96      0.93      0.95       709
           1       0.23      0.37      0.29        41

    accuracy                           0.90       750
   macro avg       0.60      0.65      0.62       750
weighted avg       0.92      0.90      0.91       750

              precision    recall  f1-score   support

           0       0.96      0.95      0.96       709
     

## **Masked**

In [ ]:
MODEL_NAME = MASKED_MMB

### *Model Setup*

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label={v:k for k,v in label2id.items()}
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=160):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx]); label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/Toxic language in football Research/models/mmBERT
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### *Dataset Split*

In [ ]:
train_ds = CommentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LEN)

# Sampler (train only)
if USE_SAMPLER:
    y_train = train_df["label_id"].values
    class_sample_count = np.bincount(y_train)     # e.g., [#non, #tox]
    weights = 1.0 / class_sample_count
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SZ)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SZ)

### *Evaluation Setup*

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device, report=False, threshold=None):
    model.eval()
    all_probs, all_preds, all_true = [], [], []

    running_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        running_loss += loss.item()

        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy())

        preds = torch.argmax(logits, dim=1) if threshold is None else (probs >= threshold).long()
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    avg_loss = running_loss / max(1, len(dataloader))

    acc       = accuracy_score(all_true, all_preds)
    f1_macro  = f1_score(all_true, all_preds, average="macro",  zero_division=0)
    f1_toxic  = f1_score(all_true, all_preds, pos_label=1,      zero_division=0)
    prec_tox  = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
    rec_tox   = recall_score(all_true, all_preds, pos_label=1,    zero_division=0)

    if report:
        print(classification_report(all_true, all_preds, digits=2))

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": prec_tox,
        "recall_toxic": rec_tox,
        "y_true": np.array(all_true),
        "y_probs": np.array(all_probs),
    }

def best_threshold_fbeta(y_true, p_toxic, beta=2.0):
    prec, rec, thr = precision_recall_curve(y_true, p_toxic)
    fbeta = (1+beta**2) * (prec*rec) / (beta**2 * prec + rec + 1e-9)
    if len(thr) == 0: return 0.5
    return float(thr[int(np.nanargmax(fbeta))])


### *Training*

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
num_warmup = int(WARMUP_RATIO * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup, num_training_steps=total_steps)

best_metric = -1.0
best_state = None
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    model.train(); running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))

    val_metrics = evaluate(model, val_loader, device, report=False, threshold=None)
    print(
    f"Epoch {epoch:>2} | "
    f"TrainLoss {train_loss:.4f} | "
    f"ValAcc {val_metrics['acc']:.4f} | "
    f"ValF1_macro {val_metrics['f1_macro']:.4f} | "
    f"ValF1_toxic {val_metrics['f1_toxic']:.4f} | "
    f"ValPrec_toxic {val_metrics['precision_toxic']:.4f} | "
    f"ValRec_toxic {val_metrics['recall_toxic']:.4f}"
    )
    try:
      val_auroc = roc_auc_score(val_metrics["y_true"], val_metrics["y_probs"])
      print(f"Val AUROC: {val_auroc:.4f}")
    except ValueError:
      print("Val AUROC: not defined (only one class present in y_true)")

    current = val_metrics["f1_macro"]
    if current > best_metric:
        best_metric = current; patience_ctr = 0
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best Val macro-F1: {best_metric:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state); model.to(device)
    print("Loaded best checkpoint (Val macro-F1=%.4f)" % best_metric)

Epoch  1 | TrainLoss 1.2037 | ValAcc 0.5867 | ValF1_macro 0.4665 | ValF1_toxic 0.2132 | ValPrec_toxic 0.1193 | ValRec_toxic 1.0000
Val AUROC: 0.9422
Epoch  2 | TrainLoss 0.0836 | ValAcc 0.9360 | ValF1_macro 0.7684 | ValF1_toxic 0.5714 | ValPrec_toxic 0.4571 | ValRec_toxic 0.7619
Val AUROC: 0.9404
Epoch  3 | TrainLoss 0.0467 | ValAcc 0.9560 | ValF1_macro 0.7942 | ValF1_toxic 0.6118 | ValPrec_toxic 0.6047 | ValRec_toxic 0.6190
Val AUROC: 0.9416
Epoch  4 | TrainLoss 0.0120 | ValAcc 0.9707 | ValF1_macro 0.8581 | ValF1_toxic 0.7317 | ValPrec_toxic 0.7500 | ValRec_toxic 0.7143
Val AUROC: 0.9493
Epoch  5 | TrainLoss 0.0104 | ValAcc 0.9613 | ValF1_macro 0.7668 | ValF1_toxic 0.5538 | ValPrec_toxic 0.7826 | ValRec_toxic 0.4286
Val AUROC: 0.9401
Epoch  6 | TrainLoss 0.0016 | ValAcc 0.9667 | ValF1_macro 0.8101 | ValF1_toxic 0.6377 | ValPrec_toxic 0.8148 | ValRec_toxic 0.5238
Val AUROC: 0.9461
Epoch  7 | TrainLoss 0.0029 | ValAcc 0.9667 | ValF1_macro 0.8101 | ValF1_toxic 0.6377 | ValPrec_toxic 0.81

### *Validation and Testing*

In [ ]:
# Tune threshold on VALIDATION set (Fbeta)
val_eval  = evaluate(model, val_loader, device, report=False, threshold=None)
best_thr  = best_threshold_fbeta(val_eval["y_true"], val_eval["y_probs"], beta=FBETA)
val_tuned = evaluate(model, val_loader, device, report=True, threshold=best_thr)

print("\n=== Validation (for tuning on Fbeta) ===")
print(f"Default thr=0.5 -> Val Acc: {val_eval['acc']:.4f} | Val Macro-F1: {val_eval['f1_macro']:.4f} | Val F1-toxic: {val_eval['f1_toxic']:.4f}")
print(f"Best thr={best_thr:.3f} (beta={FBETA}) -> Val Acc: {val_tuned['acc']:.4f} | Val Macro-F1: {val_tuned['f1_macro']:.4f} | Val F1-toxic: {val_tuned['f1_toxic']:.4f}")
try:
    val_auroc = roc_auc_score(val_eval["y_true"], val_eval["y_probs"])
    print(f"Val AUROC: {val_auroc:.4f}")
except ValueError:
    print("Val AUROC: not defined (only one class present)")

# FINAL TEST (use the tuned threshold)
test_eval_default = evaluate(model, test_loader, device, report=True, threshold=None)
test_eval_tuned   = evaluate(model, test_loader, device, report=True, threshold=best_thr)

print("\n=== FINAL TEST (unseen) ===")
print(f"Default thr=0.5 -> Test Acc: {test_eval_default['acc']:.4f} | Test Macro-F1: {test_eval_default['f1_macro']:.4f} | Test F1-toxic: {test_eval_default['f1_toxic']:.4f}")
print(f"Tuned thr={best_thr:.3f} (beta={FBETA}) -> Test Acc: {test_eval_tuned['acc']:.4f} | Test Macro-F1: {test_eval_tuned['f1_macro']:.4f} | Test F1-toxic: {test_eval_tuned['f1_toxic']:.4f}")
try:
    test_auroc = roc_auc_score(test_eval_default["y_true"], test_eval_default["y_probs"])
    print(f"Test AUROC: {test_auroc:.4f}")
except ValueError:
    print("Test AUROC: not defined (only one class present)")

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       708
           1       0.75      0.71      0.73        42

    accuracy                           0.97       750
   macro avg       0.87      0.85      0.86       750
weighted avg       0.97      0.97      0.97       750


=== Validation (for tuning on Fbeta) ===
Default thr=0.5 -> Val Acc: 0.9707 | Val Macro-F1: 0.8581 | Val F1-toxic: 0.7317
Best thr=0.519 (beta=2.0) -> Val Acc: 0.9707 | Val Macro-F1: 0.8581 | Val F1-toxic: 0.7317
Val AUROC: 0.9493
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       709
           1       0.48      0.24      0.32        41

    accuracy                           0.94       750
   macro avg       0.72      0.61      0.65       750
weighted avg       0.93      0.94      0.94       750

              precision    recall  f1-score   support

           0       0.96      0.98      0.97       709
     